# 06 --  Hyperparameter Tuning

## Concept
Hyperparameters control model behavior. Tuning finds optimal values that maximize validation performance.

## Mathematical Intuition
- **Grid Search**: Exhaustive search over all combinations
- **Random Search**: Random sampling --  more efficient for high-dimensional spaces
- **Optuna**: TPE (Tree-structured Parzen Estimator) --  Bayesian optimization that learns which regions of hyperparameter space perform well

## Interview Questions
1. Why is Random Search often better than Grid Search?
2. What is the danger of tuning on the test set?
3. How does Optuna's TPE sampler work differently from random search?

## Production Mapping
Optimizers are in `training/optimization.py`. In production, tuning runs in research notebooks; production training uses known parameters.


In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
np.random.seed(42)
n = 1000
X = pd.DataFrame({
    'capacity_mw': np.random.exponential(500, n),
    'region_risk': np.random.uniform(0, 1, n),
    'age_years': np.random.exponential(30, n),
    'num_connections': np.random.poisson(5, n),
})
y = (X['capacity_mw'] / 100 + X['region_risk'] * 5 + np.random.normal(0, 0.5, n)).clip(0, 3).round().astype(int).clip(0, 3)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print("Data ready for tuning")

In [ ]:
# Grid Search
from sklearn.model_selection import GridSearchCV
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
}
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3, n_jobs=-1, verbose=0)
grid.fit(X_train_s, y_train)
print(f"Grid Search best params: {grid.best_params_}")
print(f"Grid Search best CV score: {grid.best_score_:.4f}")

In [ ]:
# Random Search
from sklearn.model_selection import RandomizedSearchCV
param_dist = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [None, 5, 10, 15, 20, 30],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4],
}
random = RandomizedSearchCV(RandomForestClassifier(random_state=42), param_dist, n_iter=20, cv=3, n_jobs=-1, random_state=42)
random.fit(X_train_s, y_train)
print(f"Random Search best params: {random.best_params_}")
print(f"Random Search best CV score: {random.best_score_:.4f}")

In [ ]:
# Optuna
import optuna
from optuna.samplers import TPESampler

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
    }
    model = RandomForestClassifier(**params, random_state=42, n_jobs=-1)
    scores = cross_val_score(model, X_train_s, y_train, cv=3, n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=20)
print(f"Optuna best params: {study.best_params}")
print(f"Optuna best CV score: {study.best_value:.4f}")

In [ ]:
# Compare tuned vs default
default = RandomForestClassifier(random_state=42)
default.fit(X_train_s, y_train)
default_acc = accuracy_score(y_test, default.predict(X_test_s))
tuned = RandomForestClassifier(**study.best_params, random_state=42)
tuned.fit(X_train_s, y_train)
tuned_acc = accuracy_score(y_test, tuned.predict(X_test_s))
print(f"Default RF accuracy: {default_acc:.4f}")
print(f"Tuned RF accuracy: {tuned_acc:.4f}")
print(f"Improvement: {((tuned_acc - default_acc) / default_acc * 100):.2f}%")

## Key Takeaways
- Grid Search: exhaustive but expensive (3^4 = 81 combinations)
- Random Search: more efficient for high dimensions
- Optuna: smart sampling with TPE --  fewer trials needed
- Always tune on validation set, never on test set
- In production, known best params are used for deterministic re-training